# Name - Sayar Shrestha
### Student ID - 2407082

In [9]:
import pandas as pd
import numpy as np
import re
import string

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 2. Load Data

In this step, we load the dataset from the provided CSV file into a pandas DataFrame to explore and manipulate the data in upcoming stages.

In [10]:
data = pd.read_csv("trum_tweet_sentiment_analysis.csv")

# 3. Data Preprocessing & Cleaning 

In this section, we prepare the textual data for the machine learning model. This involves defining functions to clean the text (e.g., removing punctuation, non-alphabetic characters) and applying natural language processing techniques like stemming and stopword removal to reduce noise and standardize the dataset vocabulary.

In [11]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bhatt\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

In [13]:
def convert_to_string(text):
    return str(text)

In [14]:
def to_lowercase(text):
    return text.lower()

In [15]:
def remove_urls(text):
    return re.sub(r'http\S+|www\S+', '', text)

In [16]:
def remove_numbers(text):
    return re.sub(r'\d+', '', text)

In [17]:
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

In [18]:
def remove_extra_spaces(text):
    return text.strip()

In [19]:

def tokenize(text):
    return text.split()

In [20]:
def remove_stopwords(words):
    return [word for word in words if word not in stop_words]

In [21]:
def apply_stemming(words):
    return [stemmer.stem(word) for word in words]

In [22]:
def join_words(words):
    return " ".join(words)

# 4. Main Cleaning Pipeline

Here we combine all the individual cleaning functions into a single, cohesive processing pipeline function. This lets us easily apply the sequence of operations (lowercasing, removing numbers, removing URLs, punctuation, stopwords, and using stemming) uniformly to any block of text.

In [23]:
def clean_text(text):
    text = convert_to_string(text)
    text = to_lowercase(text)
    text = remove_urls(text)
    text = remove_numbers(text)
    text = remove_punctuation(text)
    text = remove_extra_spaces(text)

    words = tokenize(text)
    words = remove_stopwords(words)
    words = apply_stemming(words)

    return join_words(words)

# 5. Apply Cleaning

We use the main cleaning function created in the previous step and apply it directly to the dataset's text column. Afterward, we remove any sentences that became entirely empty or rows missing sentiment mappings so the algorithm can run efficiently.

In [24]:
data['clean_text'] = data['text'].apply(clean_text)

In [25]:
data = data[data['clean_text'].str.strip() != ""].dropna(subset=['Sentiment'])

# 6. Train-Test Split

To prevent overlapping learning and evaluation scenarios, we carefully split the cleaned dataset into discrete halves: one part to train our model on patterns, and one smaller part withheld to see how effectively the model responds to new data natively.

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    data['clean_text'],
    data['Sentiment'],
    test_size=0.2,
    random_state=42
)

# 7. Feature Extraction (TF-IDF)

Machine learning models only process numbers. In this phase, we convert text instances into numerical vectors known as arrays. The Term Frequency-Inverse Document Frequency algorithm effectively minimizes highly frequent but non-informative words while spotlighting unique and distinguishing ones.

In [27]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


# 8. Model Training

We initialize a simple Logistic Regression classifier because of its computational efficiency with categorical targets (like sentiment analysis). It is trained using the mapped values determined by the vectorized text outputs from the feature extraction.

In [28]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

LogisticRegression(max_iter=1000)

# 9. Evaluation

Now we examine exactly how efficiently our model learned. The prediction from the test sequences is measured against known mapping figures to verify accuracy, followed by generating a detailed classification report showcasing F1-scores, recall, and precision percentages.

In [29]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9178053315460619

Classification Report:

              precision    recall  f1-score   support

           0       0.93      0.95      0.94    248703
           1       0.89      0.85      0.87    121321

    accuracy                           0.92    370024
   macro avg       0.91      0.90      0.91    370024
weighted avg       0.92      0.92      0.92    370024



# 10. Prediction Function

To make the code accessible outside of batch testing, we write a simple unified prediction function. It accepts raw string text inputs, applies text cleaning and vectorization exactly as the model learned on, and then uses the trained algorithm to deliver a single generalized numerical sentiment code.

In [30]:
def predict_text(text):
    cleaned = clean_text(text)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)
    return prediction[0]

# 11. Test Examples

Finally, testing out our model directly! Using an arbitrary sequence of hardcoded examples written plainly in English, predicting sentiments manually via the loop displays actual accuracy under potential real-world text-stream input.

In [31]:
samples = [
    "This product is amazing and works perfectly!",
    "Worst service ever, I hate it"
]

for s in samples:
    print("\nText:", s)
    print("Prediction:", predict_text(s))


Text: This product is amazing and works perfectly!
Prediction: 1

Text: Worst service ever, I hate it
Prediction: 0
